# Decode Failure Analysis

This notebook investigates why some checkpoints have a lower conversion rate than the remote `baseline_remote` model.

It does not rely on the aggregate evaluator only. Instead, it:

1. Generates samples for a chosen set of models and qubit counts.
2. Decodes each sample individually.
3. Captures the exact failure stage (`tokenizer` or `backend`) and exception.
4. Adds lightweight structural diagnostics for suspicious tensor columns.

This should help identify whether failures come mainly from malformed token columns, invalid gate arity, bad padding behavior, or something else.

In [1]:
import ast
import os
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

import hydra
import numpy as np
import pandas as pd
import torch
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.evaluation.evaluator import SRVEvaluator
from my_genQC.inference.sampling import generate_tensors as _generate_tensors

In [2]:
# Edit only this cell.

DATASET_BASE = "./datasets/paper_quditkit"
QUBIT_COUNTS = [5, 6, 7, 8]
SEED = 1234
SAMPLES_PER_BUCKET = 100
USE_STRATIFIED = True
MAX_FAILURE_ROWS_PER_MODEL_QUBIT = 25
MAX_REPAIR_ROWS_PER_MODEL_QUBIT = 25

MODEL_SPECS = [
    # {
    #     "label": "baseline_remote",
    #     "model_dir": None,
    #     "hf_repo": "Floki00/qc_srv_3to8qubit",
    # },
    {
        "label": "baseline",
        "model_dir": "./models/trained/paper_srv_bucket",
        "hf_repo": None,
    },
    {
        "label": "cloob_rn50",
        "model_dir": "./models/trained/cloob_srv_bucket",
        "hf_repo": None,
    },
]

In [3]:
def _build_cfg(dataset_path, model_dir, hf_repo, num_samples):
    GlobalHydra.instance().clear()
    with hydra.initialize(version_base=None, config_path=str("../conf")):
        cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
    cfg = cfg["evaluation"]
    cfg.dataset = str(Path(dataset_path).expanduser().resolve())
    cfg.model_dir = str(Path(model_dir).expanduser().resolve()) if model_dir else None
    cfg.hf_repo = hf_repo
    cfg.num_samples = int(num_samples)
    cfg.max_gates = 16
    cfg.save_output = False
    cfg.wandb.enable = False
    return cfg


def _stratified_indices(dataset, samples_per_bucket, seed):
    rng = random.Random(seed)
    bucket_indices = defaultdict(list)
    for i, label in enumerate(dataset.y):
        text = str(label)
        srv = ast.literal_eval(text[text.find("["):text.find("]") + 1])
        n_entangled = sum(1 for v in srv if v == 2)
        bucket_indices[n_entangled].append(i)

    idx = []
    for bucket in sorted(bucket_indices):
        pool = bucket_indices[bucket]
        idx.extend(rng.sample(pool, min(samples_per_bucket, len(pool))))
    return idx


def _instruction_preview(instructions, max_items=8):
    out = []
    for ins in instructions.data[:max_items]:
        out.append(
            {
                "name": ins.name,
                "controls": list(ins.control_nodes),
                "targets": list(ins.target_nodes),
            }
        )
    return out


def _analyze_tensor_columns(tensor, valid_gate_ids, pad_constant):
    suspicious = []
    for t in range(tensor.shape[1]):
        col = tensor[:, t].detach().cpu()
        values = col.tolist()

        non_pad = [int(v) for v in values if int(v) != int(pad_constant)]
        non_zero = [v for v in non_pad if v != 0]
        if not non_zero:
            continue

        abs_ids = sorted({abs(v) for v in non_zero})
        pos = [i for i, v in enumerate(values) if v > 0 and v != pad_constant]
        neg = [i for i, v in enumerate(values) if v < 0]

        reasons = []
        if len(abs_ids) > 1:
            reasons.append("mixed_gate_ids")
        if any(g not in valid_gate_ids for g in abs_ids):
            reasons.append("unknown_gate_id")
        if len(neg) > 0 and len(pos) == 0:
            reasons.append("control_without_target")
        if len(pos) > 1:
            reasons.append("multiple_targets")
        if len(neg) > 1:
            reasons.append("multiple_controls")

        if reasons:
            suspicious.append(
                {
                    "t": t,
                    "values": values,
                    "abs_ids": abs_ids,
                    "num_pos": len(pos),
                    "num_neg": len(neg),
                    "reasons": reasons,
                }
            )

    return suspicious


def _dominant_gate_id(non_zero_vals):
    counts = Counter(abs(int(v)) for v in non_zero_vals)
    gate_id, _ = max(counts.items(), key=lambda item: (item[1], -item[0]))
    return int(gate_id)


def _pick_free_slot(values, banned):
    free = [
        i for i, v in enumerate(values)
        if int(v) == 0 and i not in banned
    ]
    if not free:
        return None
    anchor = int(round(sum(banned) / len(banned))) if banned else len(values) // 2
    return min(free, key=lambda i: (abs(i - anchor), i))


def _repair_column(col, vocabulary_inverse, pad_constant):
    values = [int(v) for v in col.tolist()]
    active = [i for i, v in enumerate(values) if v not in (0, int(pad_constant))]
    if not active:
        return col.clone(), []

    non_zero_vals = [values[i] for i in active]
    gate_id = _dominant_gate_id(non_zero_vals)
    if gate_id not in vocabulary_inverse:
        repaired = col.clone()
        for idx in active:
            repaired[idx] = 0
        return repaired, ['drop_unknown_gate']

    gate_name = str(vocabulary_inverse[gate_id]).lower()
    repaired = col.clone()
    actions = []

    if any(abs(values[i]) != gate_id for i in active):
        actions.append('normalize_gate_id')

    pos = [i for i in active if values[i] > 0 and abs(values[i]) == gate_id]
    neg = [i for i in active if values[i] < 0 and abs(values[i]) == gate_id]
    active_pool = [i for i in active if abs(values[i]) == gate_id] + [i for i in active if abs(values[i]) != gate_id]

    single_qubit = {'h', 'x', 'y', 'z', 's', 'sdg', 't', 'tdg', 'rx', 'ry', 'rz', 'p', 'u1', 'u2', 'u3'}
    controlled_two_qubit = {'cx', 'cnot', 'cz', 'cy', 'ch'}

    def _clear_active():
        for idx in active:
            repaired[idx] = 0

    if gate_name in single_qubit or (gate_name not in controlled_two_qubit and len(neg) == 0):
        target = pos[0] if pos else active_pool[0]
        _clear_active()
        repaired[target] = gate_id
        if len(pos) > 1:
            actions.append('trim_targets')
        if len(neg) > 0:
            actions.append('drop_controls_for_single_qubit_gate')
        if not pos:
            actions.append('promote_to_single_target')
        return repaired, actions

    control = neg[0] if neg else None
    target = pos[0] if pos else None

    if control is None:
        fallback = next((i for i in active_pool if i != target), None)
        if fallback is None:
            fallback = _pick_free_slot(values, {target} if target is not None else set())
        control = fallback
        if control is not None:
            actions.append('add_control')

    if target is None:
        fallback = next((i for i in active_pool if i != control), None)
        if fallback is None:
            fallback = _pick_free_slot(values, {control} if control is not None else set())
        target = fallback
        if target is not None:
            actions.append('add_target')

    if control is None or target is None or control == target:
        _clear_active()
        return repaired, actions + ['drop_unrepairable_column']

    _clear_active()
    repaired[control] = -gate_id
    repaired[target] = gate_id
    if len(neg) > 1:
        actions.append('trim_controls')
    if len(pos) > 1:
        actions.append('trim_targets')
    return repaired, actions


def _repair_tensor_heuristic(tensor, evaluator):
    repaired = tensor.detach().cpu().clone()
    action_counter = Counter()
    touched_cols = []
    pad_constant = len(evaluator.dataset.gate_pool) + 1

    for t in range(repaired.shape[1]):
        original_col = repaired[:, t].clone()
        new_col, actions = _repair_column(
            original_col,
            evaluator.tokenizer.vocabulary_inverse,
            pad_constant,
        )
        if actions:
            repaired[:, t] = new_col
            action_counter.update(actions)
            touched_cols.append({'t': t, 'actions': actions})

    return repaired, action_counter, touched_cols


def _decode_one(evaluator, tensor, prompt, sample_idx, qubit_count, model_label):
    tensor_cpu = tensor.detach().cpu()
    valid_gate_ids = set(evaluator.tokenizer.vocabulary_inverse.keys())
    pad_constant = len(evaluator.dataset.gate_pool) + 1
    suspicious_cols = _analyze_tensor_columns(tensor_cpu, valid_gate_ids, pad_constant)

    try:
        instructions = evaluator.tokenizer.decode(tensor_cpu)
    except Exception as exc:
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": False,
            "failure_stage": "tokenizer",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "instruction_preview": None,
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
            "tensor": tensor_cpu,
        }

    try:
        backend_obj = evaluator.simulator.backend.genqc_to_backend(instructions, place_barriers=False)
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": True,
            "failure_stage": None,
            "error_type": None,
            "error_message": None,
            "instruction_preview": _instruction_preview(instructions),
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
            "tensor": tensor_cpu,
            "backend_obj": backend_obj,
        }
    except Exception as exc:
        return {
            "model": model_label,
            "num_qubits": qubit_count,
            "sample_idx": sample_idx,
            "prompt": prompt,
            "success": False,
            "failure_stage": "backend",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "instruction_preview": _instruction_preview(instructions),
            "suspicious_cols": suspicious_cols,
            "tensor_head": tensor_cpu[:, :16].tolist(),
            "tensor": tensor_cpu,
        }


def inspect_model_qubit(model_spec, qubit_count, dataset_base, samples_per_bucket, use_stratified, seed):
    dataset_path = f"{dataset_base}/srv_{qubit_count}q_dataset"
    requested_samples = samples_per_bucket * qubit_count

    cfg = _build_cfg(dataset_path, model_spec.get("model_dir"), model_spec.get("hf_repo"), requested_samples)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    evaluator = SRVEvaluator(config=cfg)

    if use_stratified:
        idx = _stratified_indices(evaluator.dataset, samples_per_bucket, seed)
        evaluator.samples = len(idx)
        evaluator.idx = idx
        prompts = [str(evaluator.dataset.y[i]) for i in idx]
    else:
        evaluator.samples = min(requested_samples, len(evaluator.dataset.y))
        evaluator.idx = random.sample(range(len(evaluator.dataset.y)), k=evaluator.samples)
        prompts = [str(evaluator.dataset.y[i]) for i in evaluator.idx]

    tensors_out = _generate_tensors(
        pipeline=evaluator.pipeline,
        prompt=prompts,
        samples=evaluator.samples,
        system_size=evaluator.system_size,
        num_of_qubits=evaluator.num_qubits,
        max_gates=evaluator.max_gates,
        g=cfg.model_params.guidance_scale,
        auto_batch_size=cfg.model_params.auto_batch_size,
        enable_params=False,
        no_bar=False,
    )

    rows = []
    valid_backend = []
    valid_indices = []
    for sample_idx, (tensor, prompt) in enumerate(zip(tensors_out, prompts)):
        row = _decode_one(
            evaluator=evaluator,
            tensor=tensor,
            prompt=prompt,
            sample_idx=sample_idx,
            qubit_count=qubit_count,
            model_label=model_spec["label"],
        )
        rows.append(row)
        if row["success"]:
            valid_backend.append(row.pop("backend_obj"))
            valid_indices.append(sample_idx)

    err_rows = [row for row in rows if not row["success"]]
    stage_counter = Counter(row["failure_stage"] for row in err_rows)
    error_counter = Counter((row["failure_stage"], row["error_type"], row["error_message"]) for row in err_rows)

    summary = {
        "model": model_spec["label"],
        "num_qubits": qubit_count,
        "samples": len(rows),
        "valid": len(rows) - len(err_rows),
        "invalid": len(err_rows),
        "conversion_rate": (len(rows) - len(err_rows)) / len(rows) if rows else 0.0,
        "tokenizer_failures": stage_counter.get("tokenizer", 0),
        "backend_failures": stage_counter.get("backend", 0),
        "top_errors": error_counter.most_common(10),
    }

    return summary, rows, evaluator


def errors_to_frame(rows, keep_only_failures=True, max_rows=None):
    filtered = [row for row in rows if (not keep_only_failures) or (not row["success"])]
    if max_rows is not None:
        filtered = filtered[:max_rows]

    slim = []
    for row in filtered:
        suspicious_reason_counter = Counter(
            reason
            for col in row["suspicious_cols"]
            for reason in col["reasons"]
        )
        slim.append(
            {
                "model": row["model"],
                "num_qubits": row["num_qubits"],
                "sample_idx": row["sample_idx"],
                "failure_stage": row["failure_stage"],
                "error_type": row["error_type"],
                "error_message": row["error_message"],
                "num_suspicious_cols": len(row["suspicious_cols"]),
                "suspicious_reasons": dict(suspicious_reason_counter),
                "instruction_preview": row["instruction_preview"],
                "prompt": row["prompt"],
            }
        )

    return pd.DataFrame(slim)


def repair_failures(rows, evaluator):
    invalid_rows = [row for row in rows if not row['success']]
    repaired_rows = []

    for row in invalid_rows:
        repaired_tensor, action_counter, touched_cols = _repair_tensor_heuristic(row['tensor'], evaluator)
        repaired_row = _decode_one(
            evaluator=evaluator,
            tensor=repaired_tensor,
            prompt=row['prompt'],
            sample_idx=row['sample_idx'],
            qubit_count=row['num_qubits'],
            model_label=row['model'],
        )
        repaired_row['original_failure_stage'] = row['failure_stage']
        repaired_row['original_error_type'] = row['error_type']
        repaired_row['original_error_message'] = row['error_message']
        repaired_row['repair_action_counts'] = dict(action_counter)
        repaired_row['repair_touched_cols'] = touched_cols
        repaired_rows.append(repaired_row)

    valid_before = sum(1 for row in rows if row['success'])
    salvaged = sum(1 for row in repaired_rows if row['success'])
    total_samples = len(rows)
    action_counter = Counter(
        action
        for row in repaired_rows
        for action, count in row['repair_action_counts'].items()
        for _ in range(count)
    )
    remaining_errors = Counter(
        (row['failure_stage'], row['error_type'], row['error_message'])
        for row in repaired_rows
        if not row['success']
    )

    repaired_by_idx = {row['sample_idx']: row for row in repaired_rows}
    decoded_after = []
    for row in rows:
        if row['success']:
            decoded_row = _decode_one(
                evaluator=evaluator,
                tensor=row['tensor'],
                prompt=row['prompt'],
                sample_idx=row['sample_idx'],
                qubit_count=row['num_qubits'],
                model_label=row['model'],
            )
            decoded_after.append(decoded_row.get('backend_obj') if decoded_row['success'] else None)
        else:
            repaired_row = repaired_by_idx[row['sample_idx']]
            decoded_after.append(repaired_row.get('backend_obj') if repaired_row['success'] else None)

    valid_indices_after, target_srvs_after, predicted_srvs_after = evaluator.validate_and_calculate_srvs(
        decoded_after,
        save_output=False,
    )
    srv_exact_match_rate_after, acc_per_entanglement_after = evaluator.calculate_metrics(
        target_srvs_after,
        predicted_srvs_after,
    )

    summary = {
        'model': rows[0]['model'] if rows else None,
        'num_qubits': rows[0]['num_qubits'] if rows else None,
        'samples': total_samples,
        'invalid_before': len(invalid_rows),
        'salvaged': salvaged,
        'salvage_rate_of_invalid': salvaged / len(invalid_rows) if invalid_rows else 0.0,
        'conversion_before': valid_before / total_samples if total_samples else 0.0,
        'conversion_after': (valid_before + salvaged) / total_samples if total_samples else 0.0,
        'delta_conversion': salvaged / total_samples if total_samples else 0.0,
        'srv_exact_match_rate_after': srv_exact_match_rate_after,
        'acc_per_entanglement_after': dict(acc_per_entanglement_after),
        'effective_success_after': ((valid_before + salvaged) / total_samples) * srv_exact_match_rate_after if total_samples else 0.0,
        'num_valid_after': len(valid_indices_after),
        'top_repair_actions': action_counter.most_common(10),
        'top_remaining_errors': remaining_errors.most_common(10),
    }
    return summary, repaired_rows


def repairs_to_frame(rows, max_rows=None):
    if max_rows is not None:
        rows = rows[:max_rows]

    slim = []
    for row in rows:
        slim.append(
            {
                'model': row['model'],
                'num_qubits': row['num_qubits'],
                'sample_idx': row['sample_idx'],
                'original_failure_stage': row['original_failure_stage'],
                'original_error_type': row['original_error_type'],
                'original_error_message': row['original_error_message'],
                'repaired_success': row['success'],
                'repaired_failure_stage': row['failure_stage'],
                'repaired_error_type': row['error_type'],
                'repaired_error_message': row['error_message'],
                'repair_action_counts': row['repair_action_counts'],
                'repair_touched_cols': row['repair_touched_cols'],
                'instruction_preview': row['instruction_preview'],
                'prompt': row['prompt'],
            }
        )
    return pd.DataFrame(slim)

In [4]:
summaries = []
all_rows = {}
all_evaluators = {}

for spec in MODEL_SPECS:
    print(f"\n=== {spec['label']} ===")
    for q in QUBIT_COUNTS:
        summary, rows, evaluator = inspect_model_qubit(
            model_spec=spec,
            qubit_count=q,
            dataset_base=DATASET_BASE,
            samples_per_bucket=SAMPLES_PER_BUCKET,
            use_stratified=USE_STRATIFIED,
            seed=SEED,
        )
        summaries.append(summary)
        all_rows[(spec['label'], q)] = rows
        all_evaluators[(spec['label'], q)] = evaluator
        print(
            f"  {q}q  samples={summary['samples']}  valid={summary['valid']}  invalid={summary['invalid']}  "
            f"conversion={summary['conversion_rate']:.4f}  tokenizer={summary['tokenizer_failures']}  backend={summary['backend_failures']}"
        )


=== baseline ===
[INFO]: Cuda device has a capability of 8.6 (>= 8), allowing tf32 matmul.
2026-03-22 17:23:27 - quantum_diffusion.evaluation.evaluator - INFO - Running w/o wandb
2026-03-22 17:23:27 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_5q_dataset/dataset/ds_x.safetensors` onto device: cuda.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_5q_dataset/dataset/ds_y.safetensors` onto device: cuda.
[INFO]: Instantiated config_dataset from given config on cuda.
2026-03-22 17:23:29 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/paper_quditkit/srv_5q_dataset
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet` instantiated from given `config` on cuda.
[INFO]: Loading model from `models/trained/paper_srv_bucket/model.pt` onto device: cuda.
[INFO]: `genQC.models.unet_qc

  0%|          | 0/20 [00:00<?, ?it/s]

[INFO]: (generate_comp_tensors) Generated 500 tensors
  5q  samples=500  valid=475  invalid=25  conversion=0.9500  tokenizer=2  backend=23
[INFO]: Cuda device has a capability of 8.6 (>= 8), allowing tf32 matmul.
2026-03-22 17:23:44 - quantum_diffusion.evaluation.evaluator - INFO - Running w/o wandb
2026-03-22 17:23:44 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_6q_dataset/dataset/ds_x.safetensors` onto device: cuda.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_6q_dataset/dataset/ds_y.safetensors` onto device: cuda.
[INFO]: Instantiated config_dataset from given config on cuda.
2026-03-22 17:23:45 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/paper_quditkit/srv_6q_dataset
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet` instantiated from given `config` on cud

  0%|          | 0/20 [00:00<?, ?it/s]

[INFO]: (generate_comp_tensors) Generated 600 tensors
  6q  samples=600  valid=500  invalid=100  conversion=0.8333  tokenizer=16  backend=84
[INFO]: Cuda device has a capability of 8.6 (>= 8), allowing tf32 matmul.
2026-03-22 17:23:58 - quantum_diffusion.evaluation.evaluator - INFO - Running w/o wandb
2026-03-22 17:23:58 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_7q_dataset/dataset/ds_x.safetensors` onto device: cuda.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_7q_dataset/dataset/ds_y.safetensors` onto device: cuda.
[INFO]: Instantiated config_dataset from given config on cuda.
2026-03-22 17:24:00 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/paper_quditkit/srv_7q_dataset
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet` instantiated from given `config` on c

...

  0%|          | 0/20 [00:00<?, ?it/s]

[INFO]: (generate_comp_tensors) Generated 700 tensors
  7q  samples=700  valid=214  invalid=486  conversion=0.3057  tokenizer=271  backend=215
[INFO]: Cuda device has a capability of 8.6 (>= 8), allowing tf32 matmul.
2026-03-22 17:25:15 - quantum_diffusion.evaluation.evaluator - INFO - Running w/o wandb
2026-03-22 17:25:15 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_8q_dataset/dataset/ds_x.safetensors` onto device: cuda.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/paper_quditkit/srv_8q_dataset/dataset/ds_y.safetensors` onto device: cuda.
[INFO]: Instantiated config_dataset from given config on cuda.
2026-03-22 17:25:15 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/paper_quditkit/srv_8q_dataset
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet` instantiated from given `config` on

[INFO]: Loading CLOOB weights from ./models/encoders/cloob_rn50_yfcc_epoch_28.pt
[WARNING]: 340 unexpected CLOOB keys, first 10: ['logit_scale_hopfield', 'visual.conv1.weight', 'visual.bn1.weight', 'visual.bn1.bias', 'visual.bn1.running_mean', 'visual.bn1.running_var', 'visual.bn1.num_batches_tracked', 'visual.conv2.weight', 'visual.bn2.weight', 'visual.bn2.bias']
[INFO]: `my_genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder` instantiated from given `config` on cuda.
[INFO]: `my_genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder`. Found no key `save_type` in `config`. No state dict loaded.
[INFO]: `my_genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder`. Freeze model: True
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet` instantiated from given `config` on cuda.
[INFO]: Loading model from `models/trained/cloob_srv_bucket/embedder.pt` onto device: cuda.
[INFO]: `genQC.models.unet_qc.QC_Cond_UNet`. Freeze model: True
[WARNING]: The value 0 is reserved for background to

  0%|          | 0/20 [00:00<?, ?it/s]

[INFO]: (generate_comp_tensors) Generated 800 tensors
  8q  samples=800  valid=208  invalid=592  conversion=0.2600  tokenizer=274  backend=318


In [5]:
summary_df = pd.DataFrame(summaries)
display(summary_df[[
    'model', 'num_qubits', 'samples', 'valid', 'invalid', 'conversion_rate', 'tokenizer_failures', 'backend_failures'
]])

,model,num_qubits,samples,valid,invalid,conversion_rate,tokenizer_failures,backend_failures
0,baseline,5,500,475,25,0.950000,2,23
1,baseline,6,600,500,100,0.833333,16,84
2,baseline,7,700,521,179,0.744286,29,150
3,baseline,8,800,369,431,0.461250,164,267
4,cloob_rn50,5,500,330,170,0.660000,64,106
5,cloob_rn50,6,600,250,350,0.416667,180,170
6,cloob_rn50,7,700,214,486,0.305714,271,215
7,cloob_rn50,8,800,208,592,0.260000,274,318


In [6]:
for summary in summaries:
    key = (summary['model'], summary['num_qubits'])
    print(f"\n=== Top errors: {summary['model']} / {summary['num_qubits']}q ===")
    for (stage, err_type, err_msg), count in summary['top_errors'][:10]:
        print(f"[{count:4d}] {stage} | {err_type} | {err_msg}")


=== Top errors: baseline / 5q ===
[  17] backend | ValueError | Gate CNOT expects 2 qudits, got 3
[   6] backend | ValueError | Gate CNOT expects 2 qudits, got 1
[   2] tokenizer | RuntimeError | target_nodes.nelement() <= 0 but control_nodes.nelement() > 0

=== Top errors: baseline / 6q ===
[  70] backend | ValueError | Gate CNOT expects 2 qudits, got 3
[  16] tokenizer | RuntimeError | target_nodes.nelement() <= 0 but control_nodes.nelement() > 0
[  11] backend | ValueError | Gate CNOT expects 2 qudits, got 1
[   3] backend | ValueError | Gate H expects 1 qudits, got 2

=== Top errors: baseline / 7q ===
[ 121] backend | ValueError | Gate CNOT expects 2 qudits, got 3
[  29] tokenizer | RuntimeError | target_nodes.nelement() <= 0 but control_nodes.nelement() > 0
[  16] backend | ValueError | Gate CNOT expects 2 qudits, got 1
[  11] backend | ValueError | Gate H expects 1 qudits, got 2
[   2] backend | ValueError | Gate CNOT expects 2 qudits, got 4

=== Top errors: baseline / 8q ===
[ 

In [7]:
# Pick one model/qubit pair to inspect failure rows in detail.
INSPECT_MODEL = 'baseline'
INSPECT_QUBITS = 8

detail_df = errors_to_frame(
    all_rows[(INSPECT_MODEL, INSPECT_QUBITS)],
    keep_only_failures=True,
    max_rows=MAX_FAILURE_ROWS_PER_MODEL_QUBIT,
)
display(detail_df)

,model,num_qubits,sample_idx,failure_stage,error_type,error_message,num_suspicious_cols,suspicious_reasons,instruction_preview,prompt
0,baseline,8,0,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",1,{'multiple_targets': 1},"[{'name': 'cx', 'controls': [4], 'targets': [1...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
1,baseline,8,1,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",2,"{'multiple_targets': 1, 'multiple_controls': 1}","[{'name': 'cx', 'controls': [4], 'targets': [2...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
2,baseline,8,2,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",1,{'multiple_targets': 1},"[{'name': 'cx', 'controls': [6], 'targets': [0...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
3,baseline,8,3,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,1,{'control_without_target': 1},None,"Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
4,baseline,8,4,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",1,{'multiple_targets': 1},"[{'name': 'h', 'controls': [], 'targets': [0]}...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
5,baseline,8,7,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,1,"{'control_without_target': 1, 'multiple_contro...",None,"Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
6,baseline,8,8,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,2,"{'multiple_controls': 2, 'control_without_targ...",None,"Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
7,baseline,8,9,backend,ValueError,"Gate CNOT expects 2 qudits, got 1",0,{},"[{'name': 'h', 'controls': [], 'targets': [1]}...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
8,baseline,8,12,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",2,{'multiple_controls': 2},"[{'name': 'cx', 'controls': [3, 4], 'targets':...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
9,baseline,8,13,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",1,{'multiple_targets': 1},"[{'name': 'cx', 'controls': [3], 'targets': [1...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"


## Heuristic Repair Pass

This section tries a deliberately ad hoc local repair on failed samples only.

It repairs individual time columns by normalizing the dominant gate id, trimming extra controls or targets, and, for `cx`-like gates, optionally adding one missing endpoint on an empty qubit line.

This is not a real solution. It is only a way to test whether some failures are one-edit structural mistakes rather than completely broken samples.

In [8]:
repair_summaries = []
repaired_rows = {}

for key, rows in all_rows.items():
    repair_summary, repair_rows = repair_failures(rows, all_evaluators[key])
    repair_summaries.append(repair_summary)
    repaired_rows[key] = repair_rows

repair_summary_df = pd.DataFrame(repair_summaries)
display(repair_summary_df[[
    'model', 'num_qubits', 'samples', 'invalid_before', 'salvaged', 'salvage_rate_of_invalid',
    'conversion_before', 'conversion_after', 'delta_conversion', 'srv_exact_match_rate_after',
    'effective_success_after'
]])

2026-03-22 17:25:54 - quantum_diffusion.evaluation.evaluator - INFO - ==== genQC Evaluation ====
2026-03-22 17:25:54 - quantum_diffusion.evaluation.evaluator - INFO - Samples requested: 500
2026-03-22 17:25:54 - quantum_diffusion.evaluation.evaluator - INFO - Decoded circuits : 500
2026-03-22 17:25:54 - quantum_diffusion.evaluation.evaluator - INFO - Decode failures  : 0
2026-03-22 17:25:54 - quantum_diffusion.evaluation.evaluator - INFO - Calculating SRVs...
2026-03-22 17:25:55 - quantum_diffusion.evaluation.evaluator - INFO - Finished SRV calculation. Took 0.74 seconds.
2026-03-22 17:25:55 - quantum_diffusion.evaluation.evaluator - INFO - Calculating metrics...
2026-03-22 17:25:55 - quantum_diffusion.evaluation.evaluator - INFO - Exact SRV match rate: 0.9420
2026-03-22 17:25:55 - quantum_diffusion.evaluation.evaluator - INFO - 0 entangled qubits: 1.0000 acc
2026-03-22 17:25:55 - quantum_diffusion.evaluation.evaluator - INFO - 2 entangled qubits: 1.0000 acc
2026-03-22 17:25:55 - quant

,model,num_qubits,samples,invalid_before,salvaged,salvage_rate_of_invalid,conversion_before,conversion_after,delta_conversion,srv_exact_match_rate_after,effective_success_after
0,baseline,5,500,25,25,1.0,0.950000,1.0,0.050000,0.942000,0.942000
1,baseline,6,600,100,100,1.0,0.833333,1.0,0.166667,0.918333,0.918333
2,baseline,7,700,179,179,1.0,0.744286,1.0,0.255714,0.860000,0.860000
3,baseline,8,800,431,431,1.0,0.461250,1.0,0.538750,0.752500,0.752500
4,cloob_rn50,5,500,170,170,1.0,0.660000,1.0,0.340000,0.928000,0.928000
5,cloob_rn50,6,600,350,350,1.0,0.416667,1.0,0.583333,0.825000,0.825000
6,cloob_rn50,7,700,486,486,1.0,0.305714,1.0,0.694286,0.762857,0.762857
7,cloob_rn50,8,800,592,592,1.0,0.260000,1.0,0.740000,0.720000,0.720000


In [9]:
for summary in repair_summaries:
    print(f"\n=== Repair summary: {summary['model']} / {summary['num_qubits']}q ===")
    print(f"salvaged {summary['salvaged']} of {summary['invalid_before']} invalid samples")
    print(f"post-repair exact_match={summary['srv_exact_match_rate_after']:.4f}  effective_success={summary['effective_success_after']:.4f}")
    print(f"post-repair acc_per_entanglement={summary['acc_per_entanglement_after']}")
    print('top repair actions:')
    for action, count in summary['top_repair_actions'][:10]:
        print(f"[{count:4d}] {action}")
    if summary['top_remaining_errors']:
        print('top remaining errors:')
        for (stage, err_type, err_msg), count in summary['top_remaining_errors'][:5]:
            print(f"[{count:4d}] {stage} | {err_type} | {err_msg}")


=== Repair summary: baseline / 5q ===
salvaged 25 of 25 invalid samples
post-repair exact_match=0.9420  effective_success=0.9420
post-repair acc_per_entanglement={0: 1.0, 2: 1.0, 3: 0.95, 4: 0.96, 5: 0.8}
top repair actions:
[  11] trim_controls
[   7] trim_targets
[   6] add_control
[   2] add_target

=== Repair summary: baseline / 6q ===
salvaged 100 of 100 invalid samples
post-repair exact_match=0.9183  effective_success=0.9183
post-repair acc_per_entanglement={0: 1.0, 2: 0.97, 3: 0.9, 4: 0.9, 5: 0.86, 6: 0.88}
top repair actions:
[  50] trim_controls
[  36] trim_targets
[  15] add_target
[   7] add_control
[   6] normalize_gate_id

=== Repair summary: baseline / 7q ===
salvaged 179 of 179 invalid samples
post-repair exact_match=0.8600  effective_success=0.8600
post-repair acc_per_entanglement={0: 1.0, 2: 0.99, 3: 0.86, 4: 0.87, 5: 0.76, 6: 0.77, 7: 0.77}
top repair actions:
[ 102] trim_targets
[  78] trim_controls
[  29] add_target
[  13] add_control
[   8] normalize_gate_id

=== 

In [10]:
repair_detail_df = repairs_to_frame(
    repaired_rows[(INSPECT_MODEL, INSPECT_QUBITS)],
    max_rows=MAX_REPAIR_ROWS_PER_MODEL_QUBIT,
)
display(repair_detail_df)

,model,num_qubits,sample_idx,original_failure_stage,original_error_type,original_error_message,repaired_success,repaired_failure_stage,repaired_error_type,repaired_error_message,repair_action_counts,repair_touched_cols,instruction_preview,prompt
0,baseline,8,0,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,{'trim_targets': 1},"[{'t': 2, 'actions': ['trim_targets']}]","[{'name': 'cx', 'controls': [4], 'targets': [1...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
1,baseline,8,1,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,"{'trim_targets': 1, 'trim_controls': 1}","[{'t': 0, 'actions': ['trim_targets']}, {'t': ...","[{'name': 'cx', 'controls': [4], 'targets': [2...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
2,baseline,8,2,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,{'trim_targets': 1},"[{'t': 0, 'actions': ['trim_targets']}]","[{'name': 'cx', 'controls': [6], 'targets': [0...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
3,baseline,8,3,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,True,None,None,None,{'add_target': 1},"[{'t': 11, 'actions': ['add_target']}]","[{'name': 'cx', 'controls': [4], 'targets': [1...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
4,baseline,8,4,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,{'trim_targets': 1},"[{'t': 2, 'actions': ['trim_targets']}]","[{'name': 'h', 'controls': [], 'targets': [0]}...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
5,baseline,8,7,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,True,None,None,None,"{'add_target': 1, 'trim_controls': 1}","[{'t': 3, 'actions': ['add_target', 'trim_cont...","[{'name': 'cx', 'controls': [1], 'targets': [0...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
6,baseline,8,8,tokenizer,RuntimeError,target_nodes.nelement() <= 0 but control_nodes...,True,None,None,None,"{'trim_controls': 2, 'add_target': 1}","[{'t': 0, 'actions': ['trim_controls']}, {'t':...","[{'name': 'cx', 'controls': [1], 'targets': [7...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
7,baseline,8,9,backend,ValueError,"Gate CNOT expects 2 qudits, got 1",True,None,None,None,{'add_control': 1},"[{'t': 5, 'actions': ['add_control']}]","[{'name': 'h', 'controls': [], 'targets': [1]}...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
8,baseline,8,12,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,{'trim_controls': 2},"[{'t': 0, 'actions': ['trim_controls']}, {'t':...","[{'name': 'cx', 'controls': [3], 'targets': [6...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
9,baseline,8,13,backend,ValueError,"Gate CNOT expects 2 qudits, got 3",True,None,None,None,{'trim_targets': 1},"[{'t': 0, 'actions': ['trim_targets']}]","[{'name': 'cx', 'controls': [3], 'targets': [1...","Generate SRV: [1, 1, 1, 1, 1, 1, 1, 1]"
